# TM Composite Inference — PYNQ Notebook

Writes trained parameters to BRAM once at startup, then streams a 32×32 RGB image through
the inference module IP and reads back the per-class vote sums.

**Run order:**  
1. **Setup** — load overlay, connect to BRAM  
2. **Write parameters** — run once per power-on (weights + include bits → BRAM)  
3. **Run inference** — stream image, read class sums  

| GPIO | Offset | Direction | Bits |
|------|--------|-----------|------|
| CH1 | `0x000` | PL → PS (input) | bit 0 = `request` (ps_request AND spclst_request) |
| CH2 | `0x008` | PS → PL (output) | bits 4:1 = one-hot select, bit 0 = valid |

The same one-hot code drives both the patch-size selector and the specialist selector:  
`0001`=PS3/Spec0, `0010`=PS4/Spec1, `0100`=PS5/Spec2, `1000`=PS7/Spec3

| BRAM region | Words | Address (from specialist base) |
|-------------|-------|--------------------------------|
| Weights | 100 | 0 – 99 |
| Includes clause 0 | 68 | 100 – 167 |
| Includes clause 1 | 68 | 168 – 235 |
| … | … | … |
| Includes clause 9 | 68 | 712 – 779 |
| **Specialist base** | | `specialist_idx × 780` |

In [93]:
import sys, os, time, threading
import numpy as np
from pynq import Overlay, allocate, MMIO, Clocks

sys.path.insert(0, os.getcwd())

# ── Paths ─────────────────────────────────────────────────────────────────────
PIX_FILE   = 'data/pixel_input.txt'
PARAM_FILE = 'data/tm_params.npz'   # weights + include bits from training
OUT_DIR    = 'data/inference_output'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Hardware ──────────────────────────────────────────────────────────────────
# TODO: update bitstream name to your inference module bitstream
ol   = Overlay('bitstreams/tmc.bit')
dma  = ol.axi_dma_0
gpio = ol.axi_gpio_0

# BRAM controller — name must match block design (Vivado address editor)
bram_mmio = ol.axi_bram_ctrl_0.mmio

# Class sum AXI peripheral (CS_2_S_AXI_0)
cs_mmio = ol.CS_2_S_AXI_0.mmio

# CLK frequency reduction 
# Clocks.fclk0_mhz = 10.0
print(f'PL clock    : {Clocks.fclk0_mhz:.1f} MHz')

GPIO_CH1 = 0x000   # PL → PS: bit 0 = request (active high)
GPIO_CH2 = 0x008   # PS → PL: bits [4:1] = one-hot select, bit 0 = valid

print(f'Peripherals : {list(ol.ip_dict.keys())}')
print(f'BRAM size   : {bram_mmio.length} bytes  ({bram_mmio.length//4} words)')
print(f'Param file  : {PARAM_FILE}')
print('✓ Hardware initialised')

PL clock    : 50.0 MHz
Peripherals : ['axi_dma_0', 'axi_gpio_0', 'CS_2_S_AXI_0', 'processing_system7_0']
BRAM size   : 16384 bytes  (4096 words)
Param file  : data/tm_params.npz
✓ Hardware initialised


In [94]:
# ── Image constants ───────────────────────────────────────────────────────────
IMG_SIZE   = 32
NUM_PIXELS = IMG_SIZE * IMG_SIZE   # 1024
ENC_BITS   = 7

# ── TM architecture constants  (must match VHDL generics) ────────────────────
NUM_SPECIALISTS = 4
NUM_CLAUSES     = 8
NUM_CLASSES     = 10
MAX_WEIGHT      = 3
WEIGHT_BITS     = 4    # clog2(MAX_WEIGHT) + 2 = clog2(3)+2 = 4
MAX_PS          = 7
POS_BITS        = 29   # IMG_SIZE - PS0 = 32 - 3

# LIT_BITS = 2 * (2*POS_BITS + 3*MAX_PS*MAX_PS*ENC_BITS)
LIT_BITS = 2 * (2 * POS_BITS + 3 * MAX_PS * MAX_PS * ENC_BITS)   # 2174
SUM_BITS = 6   # clog2(MAX_WEIGHT * NUM_CLAUSES) + 1 = clog2(24)+1 = 6
               # 6-bit signed range [-32, +31]; max sum = ±24 → no overflow

# ── BRAM layout constants ─────────────────────────────────────────────────────
W_WORDS      = NUM_CLAUSES * NUM_CLASSES          # 80
INC_WORDS    = (LIT_BITS + 31) // 32              # 68
SPCLST_WORDS = W_WORDS + NUM_CLAUSES * INC_WORDS  # 624 words per specialist
BRAM_WORDS   = NUM_SPECIALISTS * SPCLST_WORDS     # 2496 total
BRAM_BYTES   = BRAM_WORDS * 4                     # 9984 bytes

# ── One-hot maps (same code for patch size and specialist) ────────────────────
SPCLST_SELECT = {0: 0b0001, 1: 0b0010, 2: 0b0100, 3: 0b1000}
PS_OF_SPCLST  = {0: 3,      1: 4,      2: 5,      3: 7}

print(f'LIT_BITS     : {LIT_BITS}')
print(f'SUM_BITS     : {SUM_BITS}  (range [{-(1<<(SUM_BITS-1))}, {(1<<(SUM_BITS-1))-1}])')
print(f'SPCLST_WORDS : {SPCLST_WORDS}  ({SPCLST_WORDS*4} bytes per specialist)')
print(f'BRAM needed  : {BRAM_BYTES} bytes  ({BRAM_WORDS} x 32-bit words)')
assert bram_mmio.length >= BRAM_BYTES, \
    f'BRAM too small: {bram_mmio.length} < {BRAM_BYTES} bytes'

LIT_BITS     : 2174
SUM_BITS     : 6  (range [-32, 31])
SPCLST_WORDS : 624  (2496 bytes per specialist)
BRAM needed  : 9984 bytes  (2496 x 32-bit words)


In [95]:
def _pack_n_inc(n_inc_clause):
    """
    Pack a boolean vector of shape (LIT_BITS,) into INC_WORDS uint32 values.

    Bit ordering matches the VHDL:
        bram_data_in[b]  →  n_inc_set[clause][word_in_clause*32 + b]
    i.e. literal 0 sits in bit 0 of the first word (little-endian).

    n_inc_clause[i] = True  → literal i is EXCLUDED (n_inc_in = '1')
    n_inc_clause[i] = False → literal i is INCLUDED (n_inc_in = '0')
    """
    padded = np.zeros(INC_WORDS * 32, dtype=np.uint8)
    padded[:LIT_BITS] = n_inc_clause.astype(np.uint8)
    packed = np.packbits(padded, bitorder='little')   # 272 bytes
    return packed.view(np.uint32)                     # 68 words


def write_all_params(bram, weights, n_inc):
    """
    Write all specialists' weights and include bits to BRAM.
    Call once after bitstream load; contents persist until power-off.

    Parameters
    ----------
    bram    : pynq.MMIO  (axi_bram_ctrl_0.mmio)
    weights : ndarray, shape (NUM_SPECIALISTS, NUM_CLAUSES, NUM_CLASSES), int32
              Values in [-MAX_WEIGHT, MAX_WEIGHT].  Stored as sign-extended 32-bit.
    n_inc   : ndarray, shape (NUM_SPECIALISTS, NUM_CLAUSES, LIT_BITS), bool
              True  = literal excluded (n_inc_in = '1')
              False = literal included (n_inc_in = '0')
              If your TM library gives include_bits (True=included), pass ~include_bits.
    """
    assert weights.shape == (NUM_SPECIALISTS, NUM_CLAUSES, NUM_CLASSES), \
        f'weights shape mismatch: {weights.shape}'
    assert n_inc.shape == (NUM_SPECIALISTS, NUM_CLAUSES, LIT_BITS), \
        f'n_inc shape mismatch: {n_inc.shape}'

    for s in range(NUM_SPECIALISTS):
        base_byte = s * SPCLST_WORDS * 4

        # ── Weights (W_WORDS words, one per (clause, class) pair, row-major) ──
        for c in range(NUM_CLAUSES):
            for j in range(NUM_CLASSES):
                offset = base_byte + (c * NUM_CLASSES + j) * 4
                bram.write(offset, int(weights[s, c, j]) & 0xFFFFFFFF)

        # ── Include bits (INC_WORDS words per clause, packed little-endian) ───
        for c in range(NUM_CLAUSES):
            words = _pack_n_inc(n_inc[s, c])
            clause_base = base_byte + (W_WORDS + c * INC_WORDS) * 4
            for k, w in enumerate(words):
                bram.write(clause_base + k * 4, int(w))

        ps = PS_OF_SPCLST[s]
        print(f'  ✓ Specialist {s}  (PS={ps})  '
              f'base word {s * SPCLST_WORDS},  '
              f'weights [{weights[s].min()}, {weights[s].max()}]')

    print('✓ All parameters written to BRAM')


In [96]:
# ── STARTUP CELL — run once per power-on ─────────────────────────────────────
# After this cell the BRAM holds all four specialists' parameters.
# Re-run only if you retrain the model or reprogram the bitstream.

params  = np.load(PARAM_FILE)
weights = params['weights'].astype(np.int32)   # (NUM_SPECIALISTS, NUM_CLAUSES, NUM_CLASSES)
n_inc   = params['n_inc'].astype(bool)         # (NUM_SPECIALISTS, NUM_CLAUSES, LIT_BITS)

# If your TM library stores include_bits (True = included) instead of
# n_inc (True = excluded), invert here:
#   n_inc = ~params['include_bits'].astype(bool)

print(f'Loaded  : {PARAM_FILE}')
print(f'weights : {weights.shape}  dtype={weights.dtype}')
print(f'n_inc   : {n_inc.shape}  dtype={n_inc.dtype}')
print()
print('Writing parameters to BRAM...')
t0 = time.time()
write_all_params(bram_mmio, weights, n_inc)
print(f'  ({time.time()-t0:.3f}s)')

Loaded  : data/tm_params.npz
weights : (4, 8, 10)  dtype=int32
n_inc   : (4, 8, 2174)  dtype=bool

Writing parameters to BRAM...
  ✓ Specialist 0  (PS=3)  base word 0,  weights [3, 3]
  ✓ Specialist 1  (PS=4)  base word 624,  weights [0, 0]
  ✓ Specialist 2  (PS=5)  base word 1248,  weights [-3, -3]
  ✓ Specialist 3  (PS=7)  base word 1872,  weights [-3, 3]
✓ All parameters written to BRAM
  (0.069s)


In [97]:
print("=== BRAM readback verification ===")
checks = [
    (0,       3,          "spec0 w[0][0] = +3"),
    (79*4,    3,          "spec0 w[7][9] = +3  (last weight)"),
    (80*4,    0xFFFFFFFF, "spec0 inc[0] w0 = 0xFFFFFFFF"),
    (623*4,   0x3FFFFFFF, "spec0 inc[7] w67= 0x3FFFFFFF (last)"),
    (624*4,   0,          "spec1 w[0][0] = 0"),
    (704*4,   0,          "spec1 inc[0] w0 = 0x00000000"),
    (1248*4,  0xFFFFFFFD, "spec2 w[0][0] = -3 → 0xFFFFFFFD"),
    (1328*4,  0xFFFFFFFF, "spec2 inc[0] w0 = 0xFFFFFFFF"),
]
ok = True
for offset, expected, label in checks:
    got = bram_mmio.read(offset)
    match = "✓" if got == expected else f"✗  got 0x{got:08x}"
    if got != expected: ok = False
    print(f"  word {offset//4:5d}: {label:40s} {match}")
print()
print("=== CS registers before any inference ===")
print(f"  reg0 = 0x{cs_mmio.read(0x00):08x}  (expect 0x00000000 initially)")
print(f"  reg1 = 0x{cs_mmio.read(0x04):08x}  (expect 0x00000000 initially)")

=== BRAM readback verification ===
  word     0: spec0 w[0][0] = +3                       ✓
  word    79: spec0 w[7][9] = +3  (last weight)        ✓
  word    80: spec0 inc[0] w0 = 0xFFFFFFFF             ✓
  word   623: spec0 inc[7] w67= 0x3FFFFFFF (last)      ✓
  word   624: spec1 w[0][0] = 0                        ✓
  word   704: spec1 inc[0] w0 = 0x00000000             ✓
  word  1248: spec2 w[0][0] = -3 → 0xFFFFFFFD          ✓
  word  1328: spec2 inc[0] w0 = 0xFFFFFFFF             ✓

=== CS registers before any inference ===
  reg0 = 0x00000000  (expect 0x00000000 initially)
  reg1 = 0x00000000  (expect 0x00000000 initially)


In [98]:
# ── Load pixel_input.txt ─────────────────────────────────────────────────────
# Format: one pixel per line, "C0 C1 C2" hex, top-row major.
# C0=R  C1=G  C2=B  (matches axis_pixel_in.vhd word layout [23:16],[15:8],[7:0])
with open(PIX_FILE) as f:
    px_lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]

assert len(px_lines) >= NUM_PIXELS, \
    f'Expected >= {NUM_PIXELS} pixel lines, got {len(px_lines)}'

pixel_words = np.zeros(NUM_PIXELS, dtype=np.uint32)
for i, line in enumerate(px_lines[:NUM_PIXELS]):
    c0, c1, c2 = (int(h, 16) for h in line.split()[:3])
    pixel_words[i] = (c0 << 16) | (c1 << 8) | c2

print(f'Loaded {len(px_lines)} lines from {PIX_FILE}')
print(f'First word : 0x{pixel_words[0]:08x}')
print(f'Last  word : 0x{pixel_words[-1]:08x}')
print('✓ Pixel data loaded')

Loaded 1024 lines from data/pixel_input.txt
First word : 0x00647dcd
Last  word : 0x00d9dcff
✓ Pixel data loaded


In [99]:
# ── GPIO helpers (same bit layout as patch generator) ────────────────────────
# gpio_mapper.vhd: gpio_in[4:1] = select, gpio_in[0] = valid
HANDSHAKE_TIMEOUT = 2.0    # seconds
DMA_POLL_PERIOD   = 0.005
DMA_POLL_TIMEOUT  = 60.0

def gpio_write(select_bits, valid):
    gpio.write(GPIO_CH2, int(valid) | (select_bits << 1))

def gpio_read_request():
    return gpio.read(GPIO_CH1) & 1

def _wait_request(label='request'):
    """Block until the hardware asserts request, or warn on timeout."""
    if gpio_read_request():
        return
    deadline = time.time() + HANDSHAKE_TIMEOUT
    while time.time() < deadline:
        if gpio_read_request():
            return
        time.sleep(0.001)
    print(f'  ⚠ Timeout waiting for {label}')

def _dma_send(buf):
    """Transfer a buffer via MM2S and poll until idle."""
    dma.sendchannel.start()
    dma.sendchannel.transfer(buf)
    deadline = time.time() + DMA_POLL_TIMEOUT
    while time.time() < deadline:
        if dma.sendchannel.idle:
            return
        time.sleep(DMA_POLL_PERIOD)
    print('  ⚠ MM2S: DMA send timeout')


def decode_class_sums(raw_64bit):
    """
    Unpack cs_data_out into a (NUM_CLASSES,) int8 array.
    cs_data_out layout (from clauses.vhd output_flattening):
        bits [i*SUM_BITS + SUM_BITS-1 : i*SUM_BITS] = class i sum
    SUM_BITS = 6, so class 0 is bits [5:0], class 1 is bits [11:6], etc.
    Signed interpretation: values in [-(MAX_WEIGHT*NUM_CLAUSES), +(MAX_WEIGHT*NUM_CLAUSES)].
    """
    mask = (1 << SUM_BITS) - 1
    sums = []
    for i in range(NUM_CLASSES):
        raw = (raw_64bit >> (i * SUM_BITS)) & mask
        # Sign-extend from SUM_BITS to Python int
        if raw >= (1 << (SUM_BITS - 1)):
            raw -= (1 << SUM_BITS)
        sums.append(raw)
    return np.array(sums, dtype=np.int16)


def run_inference(specialist_idx, pixel_words_buf):
    """
    Stream one image through the inference module for a single specialist.

    Returns the raw 64-bit cs_data_out value (call decode_class_sums() on it).
    The hardware will:
      S_RESET -> S_SETUP  (asserts request)
      [we send select + valid]
      S_SETUP -> S_LOAD   (reads BRAM, ~781 cycles)
      S_LOAD  -> S_ACCUMULATE (asserts patch_ready)
      [we stream pixels via DMA]
      S_ACCUMULATE -> S_CALCULATE -> S_RESET_CLAUSES -> S_SETUP
    """
    sel = SPCLST_SELECT[specialist_idx]

    # ── 1. Wait for request from hardware ─────────────────────────────────────
    _wait_request('specialist/PS request')

    # ── 2. Send specialist/PS select (one-hot) with valid pulse ───────────────
    gpio_write(sel, valid=1)
    time.sleep(0.0001)          # hold for at least 1 clock (~10 ns @ 100 MHz)
    gpio_write(0,   valid=0)   # deassert valid; hardware latches and moves to S_LOAD

    # ── 3. Wait for S_LOAD to finish (hardware reads BRAM, ~781 cycles) ───────
    # patch_ready_out goes high when S_ACCUMULATE is entered.
    # We add a small fixed delay (~10 µs @ 100 MHz >> 781 cycles = 7.8 µs).
    time.sleep(0.0002)

    # ── 4. Stream pixels via DMA MM2S ─────────────────────────────────────────
    _dma_send(pixel_words_buf)

    # ── 5. Read class sums ────────────────────────────────────────────────────
    # Wait for S_CALCULATE to complete after last patch
    # (only a few clock cycles needed; 1 ms is massively conservative)
    time.sleep(0.001)

    # Read class sums from CS_2_AXI registers
    # cs_data_out[31: 0] → slv_reg0  (classes 0–4 + lower 2 bits of class 5)
    # cs_data_out[63:32] → slv_reg1  (upper 4 bits of class 5, classes 6–9)
    reg0 = cs_mmio.read(0x00)
    reg1 = cs_mmio.read(0x04)
    raw  = (reg1 << 32) | reg0
    return raw

In [100]:
# ── Allocate contiguous DMA input buffer ─────────────────────────────────────
in_buf = allocate(shape=(NUM_PIXELS,), dtype=np.uint32)
in_buf[:] = pixel_words

print(f'in_buf : {in_buf.nbytes:>7} bytes  @ 0x{in_buf.physical_address:08x}')
print('✓ DMA buffer ready')

in_buf :    4096 bytes  @ 0x1584c000
✓ DMA buffer ready


In [101]:
# ── Run inference for all 4 specialists ──────────────────────────────────────
all_sums = np.zeros((NUM_SPECIALISTS, NUM_CLASSES), dtype=np.int16)

for s in range(NUM_SPECIALISTS):
    t0  = time.time()
    raw = run_inference(s, in_buf)
    dt  = time.time() - t0
    sums = decode_class_sums(raw)
    all_sums[s] = sums
    print(f'Specialist {s} (PS={PS_OF_SPCLST[s]})  {dt*1000:.1f} ms  |  '
          f'sums: {sums.tolist()}')

# ── Aggregate (sum across specialists) and classify ───────────────────────────
total = all_sums.sum(axis=0)
pred  = int(np.argmax(total))

print()
print(f'Aggregate class sums : {total.tolist()}')
print(f'Predicted class      : {pred}')

Specialist 0 (PS=3)  2.8 ms  |  sums: [12, 12, 12, 12, 12, 12, 12, 12, 12, 12]
Specialist 1 (PS=4)  3.6 ms  |  sums: [-6, -6, -6, -6, -6, -6, -6, -6, -6, -6]
Specialist 2 (PS=5)  2.5 ms  |  sums: [-6, -6, -6, -6, -6, -6, -6, -6, -6, -6]
Specialist 3 (PS=7)  2.7 ms  |  sums: [-8, -8, -8, -8, -8, -8, -8, -8, -8, -8]

Aggregate class sums : [-8, -8, -8, -8, -8, -8, -8, -8, -8, -8]
Predicted class      : 0


In [102]:
# Create a param set: all n_inc = True (all excluded → all clauses always fire)
# but weight = 0 → if loading is correct, sum MUST be 0 regardless of clause firing
import numpy as np
test_weights = np.zeros((4, 8, 10), dtype=np.int32)  # all zero
test_n_inc   = np.ones ((4, 8, 2174), dtype=bool)    # all excluded (always fire)
write_all_params(bram_mmio, test_weights, test_n_inc)

raw = run_inference(0, in_buf)
sums = decode_class_sums(raw)
print(f"Zero-weight, all-excluded → sums: {sums.tolist()}")
print(f"Expected: all 0. {'✓ PASS' if all(s == 0 for s in sums) else '✗ FAIL — weight loading is wrong'}")

  ✓ Specialist 0  (PS=3)  base word 0,  weights [0, 0]
  ✓ Specialist 1  (PS=4)  base word 624,  weights [0, 0]
  ✓ Specialist 2  (PS=5)  base word 1248,  weights [0, 0]
  ✓ Specialist 3  (PS=7)  base word 1872,  weights [0, 0]
✓ All parameters written to BRAM
Zero-weight, all-excluded → sums: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Expected: all 0. ✓ PASS


In [103]:
# Test: weight = +1, all INCLUDED (n_inc = False) 
# Clauses should NEVER fire → sum must be 0 regardless of image
test_w = np.ones ((4, 8, 10),   dtype=np.int32)   # all +1
test_n = np.zeros((4, 8, 2174), dtype=bool)        # all included (n_inc_in = '0')
write_all_params(bram_mmio, test_w, test_n)

for s in range(4):
    raw  = run_inference(s, in_buf)
    sums = decode_class_sums(raw)
    result = "✓ PASS" if all(x == 0 for x in sums) else f"✗ FAIL {sums}"
    print(f"Spec {s} weight=+1 all-included: {result}")

  ✓ Specialist 0  (PS=3)  base word 0,  weights [1, 1]
  ✓ Specialist 1  (PS=4)  base word 624,  weights [1, 1]
  ✓ Specialist 2  (PS=5)  base word 1248,  weights [1, 1]
  ✓ Specialist 3  (PS=7)  base word 1872,  weights [1, 1]
✓ All parameters written to BRAM
Spec 0 weight=+1 all-included: ✓ PASS
Spec 1 weight=+1 all-included: ✓ PASS
Spec 2 weight=+1 all-included: ✓ PASS
Spec 3 weight=+1 all-included: ✓ PASS


In [104]:
# Distinguishable weights: clause c gets weight = c - 3 → {-3, -2, -1, 0, 1, 2, 3, 3}
# If ALL 8 fire: sum per class = -3-2-1+0+1+2+3+3 = +3
# Each subset that fires gives a unique sum, so we can identify exactly which clauses fired
clause_weights = np.array([0, 0, 0, 0, 0, 0, 0, 1], dtype=np.int32)
tw = np.zeros((4, 8, 10), dtype=np.int32)
for c in range(8):
    tw[:, c, :] = clause_weights[c]
tn = np.ones((4, 8, 2174), dtype=bool)   # all excluded → every loaded clause should fire

write_all_params(bram_mmio, tw, tn)
for s in range(4):
    raw  = run_inference(s, in_buf)
    sums = decode_class_sums(raw)
    print(f"Spec {s}: sums={sums.tolist()[0]} (expect +3 if all 8 fire)")

  ✓ Specialist 0  (PS=3)  base word 0,  weights [0, 1]
  ✓ Specialist 1  (PS=4)  base word 624,  weights [0, 1]
  ✓ Specialist 2  (PS=5)  base word 1248,  weights [0, 1]
  ✓ Specialist 3  (PS=7)  base word 1872,  weights [0, 1]
✓ All parameters written to BRAM
Spec 0: sums=0 (expect +3 if all 8 fire)
Spec 1: sums=-6 (expect +3 if all 8 fire)
Spec 2: sums=-6 (expect +3 if all 8 fire)
Spec 3: sums=-8 (expect +3 if all 8 fire)


In [105]:
# 1) Print all 10 classes per specialist
clause_weights = np.array([0, 0, 0, 0, 0, 0, 0, 1], dtype=np.int32)
tw = np.zeros((4, 8, 10), dtype=np.int32)
for c in range(8):
    tw[:, c, :] = clause_weights[c]
tn = np.ones((4, 8, 2174), dtype=bool)
write_all_params(bram_mmio, tw, tn)

for s in range(4):
    raw  = run_inference(s, in_buf)
    sums = decode_class_sums(raw)
    print(f"Spec {s}: {sums.tolist()}")

# 2) Verify the BRAM actually contains what Python wrote
print("\n--- BRAM weight readback for Spec 0 (expect words 0-9=-3, 10-19=-2, ..., 70-79=+3) ---")
for c in range(8):
    word_idx = c * 10  # first class slot for clause c
    raw = bram_mmio.read(word_idx * 4)
    w4  = raw & 0xF
    val = w4 - 16 if w4 >= 8 else w4   # 4-bit signed
    print(f"  clause {c}, word {word_idx}: raw=0x{raw:08X}, lower 4 bits=0x{w4:X}, signed4={val}, expected={clause_weights[c]}")

  ✓ Specialist 0  (PS=3)  base word 0,  weights [0, 1]
  ✓ Specialist 1  (PS=4)  base word 624,  weights [0, 1]
  ✓ Specialist 2  (PS=5)  base word 1248,  weights [0, 1]
  ✓ Specialist 3  (PS=7)  base word 1872,  weights [0, 1]
✓ All parameters written to BRAM
Spec 0: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Spec 1: [-6, -6, -6, -6, -6, -6, -6, -6, -6, -6]
Spec 2: [-6, -6, -6, -6, -6, -6, -6, -6, -6, -6]
Spec 3: [-8, -8, -8, -8, -8, -8, -8, -8, -8, -8]

--- BRAM weight readback for Spec 0 (expect words 0-9=-3, 10-19=-2, ..., 70-79=+3) ---
  clause 0, word 0: raw=0x00000000, lower 4 bits=0x0, signed4=0, expected=0
  clause 1, word 10: raw=0x00000000, lower 4 bits=0x0, signed4=0, expected=0
  clause 2, word 20: raw=0x00000000, lower 4 bits=0x0, signed4=0, expected=0
  clause 3, word 30: raw=0x00000000, lower 4 bits=0x0, signed4=0, expected=0
  clause 4, word 40: raw=0x00000000, lower 4 bits=0x0, signed4=0, expected=0
  clause 5, word 50: raw=0x00000000, lower 4 bits=0x0, signed4=0, expected=0
  cl

In [106]:
# Test each clause individually: weight=+3 on one clause only, n_inc=True everywhere.
# Each clause that correctly loads + fires contributes +3. Each that fails contributes 0.
tn = np.ones((4, 8, 2174), dtype=bool)   # all excluded → any working clause must fire

print(f"{'clause':<8}{'spec0':>8}{'spec1':>8}{'spec2':>8}{'spec3':>8}    (expect +3 if fires)")
for clause_test in range(8):
    tw = np.zeros((4, 8, 10), dtype=np.int32)
    tw[:, clause_test, :] = 3        # only this clause gets +3
    write_all_params(bram_mmio, tw, tn)
    
    row = [clause_test]
    for s in range(4):
        raw  = run_inference(s, in_buf)
        sums = decode_class_sums(raw)
        row.append(int(sums[0]))     # all classes identical, pick class 0
    print(f"  {row[0]:<6}{row[1]:>8}{row[2]:>8}{row[3]:>8}{row[4]:>8}")

clause     spec0   spec1   spec2   spec3    (expect +3 if fires)
  ✓ Specialist 0  (PS=3)  base word 0,  weights [0, 3]
  ✓ Specialist 1  (PS=4)  base word 624,  weights [0, 3]
  ✓ Specialist 2  (PS=5)  base word 1248,  weights [0, 3]
  ✓ Specialist 3  (PS=7)  base word 1872,  weights [0, 3]
✓ All parameters written to BRAM
  0            0      -6      -6      -8
  ✓ Specialist 0  (PS=3)  base word 0,  weights [0, 3]
  ✓ Specialist 1  (PS=4)  base word 624,  weights [0, 3]
  ✓ Specialist 2  (PS=5)  base word 1248,  weights [0, 3]
  ✓ Specialist 3  (PS=7)  base word 1872,  weights [0, 3]
✓ All parameters written to BRAM
  1           12      -6      -6      -8
  ✓ Specialist 0  (PS=3)  base word 0,  weights [0, 3]
  ✓ Specialist 1  (PS=4)  base word 624,  weights [0, 3]
  ✓ Specialist 2  (PS=5)  base word 1248,  weights [0, 3]
  ✓ Specialist 3  (PS=7)  base word 1872,  weights [0, 3]
✓ All parameters written to BRAM
  2            0      -6      -6      -8
  ✓ Specialist 0  (PS=3)  bas

In [107]:
# Re-write current per-clause weights, then read back ALL four specs
clause_weights = np.array([-3, -2, -1, 0, 1, 2, 3, 3], dtype=np.int32)
tw = np.zeros((4, 8, 10), dtype=np.int32)
for c in range(8):
    tw[:, c, :] = clause_weights[c]
tn = np.ones((4, 8, 2174), dtype=bool)
write_all_params(bram_mmio, tw, tn)

SPCLST_WORDS = 624
NUM_CLAUSES = 8
NUM_CLASSES = 10

for s in range(4):
    base = s * SPCLST_WORDS
    print(f"\nSpec {s} weight readback (base word {base}, expect c0=-3 ... c7=+3):")
    for c in range(NUM_CLAUSES):
        word_idx = base + c * NUM_CLASSES
        raw = bram_mmio.read(word_idx * 4)
        w4  = raw & 0xF
        val = w4 - 16 if w4 >= 8 else w4
        ok  = "✓" if val == clause_weights[c] else "✗"
        print(f"  {ok} clause {c}, word {word_idx}: raw=0x{raw:08X}, signed4={val}, expected={clause_weights[c]}")

  ✓ Specialist 0  (PS=3)  base word 0,  weights [-3, 3]
  ✓ Specialist 1  (PS=4)  base word 624,  weights [-3, 3]
  ✓ Specialist 2  (PS=5)  base word 1248,  weights [-3, 3]
  ✓ Specialist 3  (PS=7)  base word 1872,  weights [-3, 3]
✓ All parameters written to BRAM

Spec 0 weight readback (base word 0, expect c0=-3 ... c7=+3):
  ✓ clause 0, word 0: raw=0xFFFFFFFD, signed4=-3, expected=-3
  ✓ clause 1, word 10: raw=0xFFFFFFFE, signed4=-2, expected=-2
  ✓ clause 2, word 20: raw=0xFFFFFFFF, signed4=-1, expected=-1
  ✓ clause 3, word 30: raw=0x00000000, signed4=0, expected=0
  ✓ clause 4, word 40: raw=0x00000001, signed4=1, expected=1
  ✓ clause 5, word 50: raw=0x00000002, signed4=2, expected=2
  ✓ clause 6, word 60: raw=0x00000003, signed4=3, expected=3
  ✓ clause 7, word 70: raw=0x00000003, signed4=3, expected=3

Spec 1 weight readback (base word 624, expect c0=-3 ... c7=+3):
  ✓ clause 0, word 624: raw=0xFFFFFFFD, signed4=-3, expected=-3
  ✓ clause 1, word 634: raw=0xFFFFFFFE, signed4=-2

In [108]:
# ── Helper: save parameters from a trained TM model ──────────────────────────
# Run this on the HOST after training, then copy tm_params.npz to the PYNQ.
#
# For GraphTsetlinMachine (or similar) the include state array has shape
# (num_classes * clauses_per_class, 2 * num_features).  Adapt to your library.
#
# Convention used by this notebook:
#   n_inc[s, c, l] = True  →  literal l is EXCLUDED from clause c of specialist s
#   n_inc[s, c, l] = False →  literal l is INCLUDED
#
# Example (PyTsetlinMachine ta_state shape):
# ----------------------------------------------------------------------
# def save_params(models, out_path):
#     """
#     models : list of 4 trained TM models, one per specialist.
#     Each model.ta_state has shape (NUM_CLAUSES, LIT_BITS) with
#     positive = include, negative/zero = exclude.
#     """
#     weights = np.zeros((NUM_SPECIALISTS, NUM_CLAUSES, NUM_CLASSES), dtype=np.int32)
#     n_inc   = np.ones( (NUM_SPECIALISTS, NUM_CLAUSES, LIT_BITS),   dtype=bool)
#
#     THRESHOLD = 0   # automaton state > THRESHOLD means include
#     for s, model in enumerate(models):
#         weights[s] = model.weight_matrix.astype(np.int32)  # shape (C, K)
#         n_inc[s]   = model.ta_state <= THRESHOLD            # True = excluded
#
#     np.savez(out_path, weights=weights, n_inc=n_inc)
#     print(f'Saved {out_path}')
# ----------------------------------------------------------------------
print('See cell source for the save_params() helper.')
print('Copy data/tm_params.npz to the PYNQ before running cell_bram_write.')

See cell source for the save_params() helper.
Copy data/tm_params.npz to the PYNQ before running cell_bram_write.
